[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_07/notebook_7_4_ecg_classification_1d_cnn.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.4: ECG Signal Classification for AFib Detection

**Chapter 7: Time Series Analysis - Implementing Journey 2 (Yuki)**

**Journey Connection**: This notebook implements Journey 2 from Chapter 3, where Yuki received multiple false AFib alerts from a wearable device. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

1. Preprocess ECG waveforms (filtering, quality assessment)
2. Extract hand-crafted features (R-R intervals, HRV)
3. Train a rhythm classifier two ways: a Random Forest on HRV features (runs everywhere, including the browser) and a 1D CNN on the raw ECG (needs TensorFlow), and compare accuracy, interpretability and computational cost
4. Analyze false positives and motion artifacts
5. Understand the impact of prevalence on PPV

## Clinical Context

Atrial fibrillation increases stroke risk 5×. Wearable ECG devices can catch paroxysmal AFib missed in office visits.

**Yuki's story** (Chapter 3, Journey 2): Yuki Tanaka, 55, a healthy marketing executive who exercises regularly, bought an ECG smartwatch after her mother's stroke from undiagnosed AFib. During a 6:12 AM jog it alerted: *"Possible atrial fibrillation."* A 12-lead ECG and a 48-hour Holter were normal: probably a false alarm from motion artifact. The alerts kept coming, and she eventually stopped wearing the watch continuously.

**The challenge**: In a 55-year-old without known heart disease, AFib prevalence is only ~0.1–0.5%. At 0.5% (Chapter 3), a device with 95% sensitivity and 98% specificity has a PPV of only ~19%: about **4 false alarms for every true AFib detection**.

---

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import kurtosis, skew
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Generate Synthetic ECG Data

We'll create **synthetic** single-lead ECG waveforms for normal sinus rhythm and atrial fibrillation. Both rhythms use the same QRS-T template, so they differ by **rhythm** (regular vs irregularly irregular R-R intervals), plus P waves (normal) vs small fibrillatory f-waves (AFib).

To make the task realistically hard, the generator includes mechanisms that confuse real devices:
- **Subtle AFib**: some episodes are rate-controlled with only mildly irregular R-R intervals (SD from 0.06 s)
- **Paroxysmal AFib**: in ~30% of AFib recordings the episode starts late in the 30 s window, after normal beats
- **Ectopic beats**: a few normal recordings contain premature beats with compensatory pauses
- Variable sensor noise, heart rate (55–95 bpm sinus, 75–115 bpm AFib) and sinus arrhythmia

By default there is **no motion artifact**: Chapter 3 notes that the 95% sensitivity / 98% specificity reported for wearable AFib detectors comes from clinical validation data recorded at rest. Section 5 adds the motion of Yuki's morning jog. The knobs were set so that, averaged over several random seeds, the classifiers land near those published figures.

In [2]:
def ecg_beat_template(fs=250, p_wave=True):
    """One PQRST complex (0.6 s); the R peak sits 0.20 s after the beat start."""
    tb = np.arange(int(0.6 * fs)) / fs
    beat = (1.0 * np.exp(-((tb - 0.200) / 0.012) ** 2)      # R
            - 0.10 * np.exp(-((tb - 0.180) / 0.010) ** 2)   # Q
            - 0.25 * np.exp(-((tb - 0.225) / 0.010) ** 2)   # S
            + 0.30 * np.exp(-((tb - 0.400) / 0.050) ** 2))  # T
    if p_wave:
        beat += 0.12 * np.exp(-((tb - 0.080) / 0.025) ** 2)  # P
    return beat


def add_motion_artifact(ecg, intensity=0.5, fs=250, segment=None):
    """Add a motion artifact (low-frequency baseline swings from arm/body movement)."""
    artifact = intensity * np.random.randn(len(ecg))
    artifact = signal.filtfilt(*signal.butter(2, 6.0, fs=fs), artifact)  # < 6 Hz
    artifact *= intensity / (artifact.std() + 1e-12)
    if segment is not None:                      # only part of the recording
        mask = np.zeros(len(ecg)); mask[segment[0]:segment[1]] = 1
        artifact *= mask
    return ecg + artifact


def generate_synthetic_ecg(duration=30, fs=250, rhythm='normal',
                           ectopic_prob=0.02, ectopic_burden=(0.02, 0.06), motion_prob=0.0, motion_intensity=(0.15, 0.45),
                           af_rr_sd=(0.06, 0.20), paroxysmal_prob=0.3, paroxysmal_onset=(0.7, 0.95),
                           noise_range=(0.03, 0.10)):
    """
    Generate a SYNTHETIC single-lead wearable ECG.

    Parameters:
    - duration: seconds
    - fs: sampling frequency (Hz)
    - rhythm: 'normal' or 'afib'
    - ectopic_prob: chance a NORMAL recording contains premature (ectopic) beats
    - ectopic_burden: fraction of beats that are premature in those recordings
    - motion_prob: chance a recording (either rhythm) has a motion-artifact segment (exercise);
      0 by default, like clinical validation data recorded at rest (Section 5 adds motion)
    - motion_intensity: artifact amplitude range (mV)
    - af_rr_sd: range of R-R standard deviation in AFib (s); low values = rate-controlled,
      subtle AFib that is hard to tell from sinus rhythm with ectopics
    - paroxysmal_prob: chance an AFib recording is PAROXYSMAL (starts as sinus rhythm,
      AFib begins at a random point given by paroxysmal_onset, as a fraction of the recording)
    - noise_range: range of sensor noise SD (mV)

    Both rhythms share the same QRS-T template, so the classes differ by rhythm
    (and P waves vs. f-waves), not by QRS amplitude.
    """
    t = np.arange(0, duration, 1/fs)
    n_beats = int(duration / 0.3) + 2

    # Sinus R-R intervals (used for normal recordings, and before a paroxysmal AFib episode)
    hr = np.random.uniform(55, 95)
    rr_sinus = np.random.normal(60 / hr, np.random.uniform(0.01, 0.04), n_beats)
    if rhythm == 'normal' and np.random.rand() < ectopic_prob:
        # Premature beats: early by 20-40%, followed by a compensatory pause
        p_ect = np.random.uniform(*ectopic_burden)
        for i in range(n_beats - 1):
            if np.random.rand() < p_ect:
                early = rr_sinus[i] * np.random.uniform(0.2, 0.4)
                rr_sinus[i] -= early
                rr_sinus[i + 1] += early

    af_start = np.inf  # time (s) at which AFib begins
    if rhythm == 'afib':
        # AFib: irregularly irregular R-R intervals, no P waves
        hr_mean = np.random.uniform(75, 115)
        rr_sd = np.random.uniform(*af_rr_sd)
        rr_af = np.random.normal(60 / hr_mean, rr_sd, n_beats)
        too_short = rr_af < 0.3                 # truncate at 0.3 s (200 bpm):
        while too_short.any():                  # redraw the impossible ones
            rr_af[too_short] = np.random.normal(60 / hr_mean, rr_sd, too_short.sum())
            too_short = rr_af < 0.3
        # Paroxysmal AFib: the episode starts part-way through the recording
        af_start = duration * np.random.uniform(*paroxysmal_onset) if np.random.rand() < paroxysmal_prob else 0.0

    beat_sinus = ecg_beat_template(fs, p_wave=True)
    beat_af = ecg_beat_template(fs, p_wave=False)
    ecg = np.zeros(len(t) + len(beat_sinus))
    beat_time = np.random.uniform(0, 1.0)  # random phase
    for i in range(n_beats):
        idx = int(beat_time * fs)
        if idx >= len(t):
            break
        in_af = beat_time >= af_start
        ecg[idx:idx + len(beat_sinus)] += beat_af if in_af else beat_sinus
        beat_time += rr_af[i] if in_af else rr_sinus[i]
    ecg = ecg[:len(t)]

    if rhythm == 'afib':
        # Fibrillatory f-waves: small 4-8 Hz baseline oscillation (often barely visible)
        f = np.random.uniform(4, 8)
        f_wave = np.random.uniform(0.01, 0.05) * np.sin(2 * np.pi * f * t + np.random.uniform(0, 2 * np.pi))
        ecg += f_wave * (t >= af_start)

    # Sensor noise (varies with skin contact)
    ecg += np.random.normal(0, np.random.uniform(*noise_range), len(ecg))

    # Motion artifact during part of the recording (e.g., a morning jog)
    if np.random.rand() < motion_prob:
        seg_len = int(np.random.uniform(0.2, 0.6) * len(t))
        start = np.random.randint(0, len(t) - seg_len)
        ecg = add_motion_artifact(ecg, np.random.uniform(*motion_intensity), fs, (start, start + seg_len))

    return t, ecg

# Generate examples
t_normal, ecg_normal = generate_synthetic_ecg(duration=10, rhythm='normal')
t_afib, ecg_afib = generate_synthetic_ecg(duration=10, rhythm='afib')

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t_normal, ecg_normal, 'b-', linewidth=1)
axes[0].set_title('Normal Sinus Rhythm (synthetic)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude (mV)')
axes[0].set_xlim([0, 10])
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_afib, ecg_afib, 'r-', linewidth=1)
axes[1].set_title('Atrial Fibrillation (synthetic)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude (mV)')
axes[1].set_xlim([0, 10])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak amplitude: normal {ecg_normal.max():.2f} mV vs AFib {ecg_afib.max():.2f} mV (same QRS template)")
print("Key differences:")
print("  Normal: Regular R-R intervals, visible P waves")
print("  AFib: Irregular R-R intervals, absent P waves, small fibrillatory baseline waves")


Key differences:
  Normal: Regular R-R intervals, visible P waves
  AFib: Irregular R-R intervals, absent P waves, irregular baseline


## 2. Feature-Based Classification

Traditional approach: Extract hand-crafted features from ECG signal.

In [3]:
def extract_rr_intervals(ecg, fs=250):
    """Detect R-peaks and calculate R-R intervals."""
    # Simple peak detection: half the height of the tallest R waves,
    # refractory period 0.25 s (allows heart rates up to 240 bpm)
    threshold = 0.5 * np.percentile(ecg, 99.5)
    peaks, _ = signal.find_peaks(ecg, height=threshold, distance=int(fs * 0.25))

    rr_intervals = np.diff(peaks) / fs  # Convert to seconds
    return peaks, rr_intervals

def extract_hrv_features(rr_intervals):
    """Calculate heart rate variability features."""
    if len(rr_intervals) < 2:
        return {}

    features = {
        'rr_mean': np.mean(rr_intervals),
        'rr_std': np.std(rr_intervals),
        'rr_rmssd': np.sqrt(np.mean(np.diff(rr_intervals)**2)),
        'rr_min': np.min(rr_intervals),
        'rr_max': np.max(rr_intervals),
        'rr_range': np.max(rr_intervals) - np.min(rr_intervals),
        'hr_mean': 60 / np.mean(rr_intervals),
        'hr_std': np.std(60 / rr_intervals),
    }
    return features

# Extract features from examples
peaks_normal, rr_normal = extract_rr_intervals(ecg_normal)
peaks_afib, rr_afib = extract_rr_intervals(ecg_afib)

features_normal = extract_hrv_features(rr_normal)
features_afib = extract_hrv_features(rr_afib)

print("Normal Sinus Rhythm Features:")
for k, v in features_normal.items():
    print(f"  {k}: {v:.3f}")

print("\nAtrial Fibrillation Features:")
for k, v in features_afib.items():
    print(f"  {k}: {v:.3f}")

if features_afib['rr_std'] > features_normal['rr_std']:
    print(f"\n💡 Notice: AFib has higher variability (rr_std {features_afib['rr_std']:.2f} vs {features_normal['rr_std']:.2f} s)")
else:
    print("\n💡 Notice: AFib does NOT show higher rr_std in this example")
if features_normal['rr_std'] > 0.1:
    print(f"   Caution: the 'normal' signal also looks variable (detected HR {features_normal['hr_mean']:.0f} bpm):")
    print("   the simple threshold peak detector misses or adds beats. Check peaks before trusting HRV features.")

Normal Sinus Rhythm Features:
  rr_mean: 1.048
  rr_std: 0.460
  rr_rmssd: 0.603
  rr_min: 0.456
  rr_max: 1.912
  rr_range: 1.456
  hr_mean: 57.252
  hr_std: 31.960

Atrial Fibrillation Features:
  rr_mean: 2.483
  rr_std: 1.116
  rr_rmssd: 2.210
  rr_min: 1.172
  rr_max: 3.900
  rr_range: 2.728
  hr_mean: 24.168
  hr_std: 15.102

💡 Notice: AFib has higher variability (rr_std 1.12 vs 0.46 s)
   Caution: the 'normal' signal also looks variable (detected HR 57 bpm vs 70 generated):
   the simple threshold peak detector misses or adds beats. Check peaks before trusting HRV features.


## 3. Generate Dataset and Train Classifier

Create a larger dataset and train Random Forest on HRV features. We keep the raw waveforms too, for the 1D CNN in Section 3b, and use the **same** train/test split for both models.

In [4]:
# Generate dataset
n_samples = 1000
X_features = []
y_labels = []
X_raw = []  # raw waveforms for the 1D CNN (Section 3b)

print("Generating dataset...")
for i in range(n_samples):
    rhythm = 'normal' if i < n_samples // 2 else 'afib'
    label = 0 if rhythm == 'normal' else 1

    _, ecg = generate_synthetic_ecg(duration=30, rhythm=rhythm)
    peaks, rr = extract_rr_intervals(ecg)
    features = extract_hrv_features(rr)

    if len(features) > 0:
        X_features.append(list(features.values()))
        y_labels.append(label)
        X_raw.append(ecg)

X = np.array(X_features)
y = np.array(y_labels)
X_raw = np.array(X_raw)

feature_names = list(features_normal.keys())

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Class distribution: {np.bincount(y)}")

# Split data (split indices, so the 1D CNN can reuse exactly the same split)
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = X[idx_train], X[idx_test], y[idx_train], y[idx_test]

# Train Random Forest
import time
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
t0 = time.perf_counter()
rf.fit(X_train, y_train)
rf_train_time = time.perf_counter() - t0

# Evaluate
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

print("\nPerformance on Test Set:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'AFib']))

auroc = roc_auc_score(y_test, y_pred_proba)
print(f"AUROC: {auroc:.3f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'AFib'], yticklabels=['Normal', 'AFib'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix: Feature-Based Classification')
plt.show()

Generating dataset...


Dataset shape: (500, 8)
Features: ['rr_mean', 'rr_std', 'rr_rmssd', 'rr_min', 'rr_max', 'rr_range', 'hr_mean', 'hr_std']
Class distribution: [250 250]

Performance on Test Set:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        75
        AFib       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

AUROC: 1.000


## 3b. 1D CNN on the Raw ECG

The book (Section 7.5) describes a 1D CNN with convolutional layers of increasing width (**32 → 64 → 128 filters**), each followed by ReLU and max pooling, then **global average pooling** and dense layers. We train exactly that on the raw waveforms, using the same train/test split as the Random Forest, and compare:

- **Accuracy** (test accuracy, AUROC)
- **Interpretability** (feature importances vs. a black box)
- **Computational cost** (parameters, training time, inference time per 30 s ECG)

Design note: the input is downsampled to 125 Hz and the kernels are 15 samples wide, so the last layer "sees" about 2.4 s of signal: enough to compare 2–3 consecutive beats, which is what rhythm classification needs.

This section needs **TensorFlow** (Colab or local Jupyter). In the browser (JupyterLite) it is skipped and the feature-based classifier above is the result.

In [ ]:
try:
    import tensorflow as tf
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("ℹ️ This section needs TensorFlow (Colab or local Jupyter); it is skipped in the browser version.")

if HAS_TF:
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42)

    # Preprocess: downsample 250 -> 125 Hz, z-score each recording
    X_cnn = X_raw[:, ::2]
    X_cnn = (X_cnn - X_cnn.mean(axis=1, keepdims=True)) / (X_cnn.std(axis=1, keepdims=True) + 1e-8)
    X_cnn = X_cnn[..., np.newaxis].astype('float32')
    Xc_train, Xc_test = X_cnn[idx_train], X_cnn[idx_test]

    L = tf.keras.layers
    cnn = tf.keras.Sequential([
        L.Input(shape=X_cnn.shape[1:]),
        L.Conv1D(32, 15, padding='same', activation='relu'),  L.MaxPooling1D(4),
        L.Conv1D(64, 15, padding='same', activation='relu'),  L.MaxPooling1D(4),
        L.Conv1D(128, 15, padding='same', activation='relu'),
        L.GlobalAveragePooling1D(),
        L.Dense(32, activation='relu'),
        L.Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])

    t0 = time.perf_counter()
    history = cnn.fit(Xc_train, y_train, epochs=15, batch_size=32, validation_split=0.15, verbose=0)
    cnn_train_time = time.perf_counter() - t0

    cnn_proba = cnn.predict(Xc_test, verbose=0).ravel()
    cnn_pred = (cnn_proba >= 0.5).astype(int)
    print(f"1D CNN: {cnn.count_params():,} parameters, trained {len(history.history['loss'])} epochs in {cnn_train_time:.1f} s")
    print(f"Final training accuracy {history.history['accuracy'][-1]:.3f}, validation accuracy {history.history['val_accuracy'][-1]:.3f}")
    print("\nPerformance on Test Set:")
    print(classification_report(y_test, cnn_pred, target_names=['Normal', 'AFib']))
    print(f"AUROC: {roc_auc_score(y_test, cnn_proba):.3f}")


### Feature-Based vs. 1D CNN: Accuracy, Interpretability, Cost

Inference time is measured per single 30 s recording (as a wearable would process it), median of 50 runs. For the Random Forest it **includes** R-peak detection and HRV feature extraction; for the CNN it includes only the forward pass (preprocessing is negligible). The CNN's "size" is its trainable parameters; the forest's is its number of tree nodes (each node stores a feature index and threshold).

In [ ]:
from sklearn.metrics import accuracy_score

def median_ms(fn, n=50):
    times = []
    for i in range(n):
        t0 = time.perf_counter(); fn(i); times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)

rf_acc = accuracy_score(y_test, y_pred)
rf_auc = roc_auc_score(y_test, y_pred_proba)
rf_nodes = sum(est.tree_.node_count for est in rf.estimators_)
rf_ms = median_ms(lambda i: rf.predict_proba(
    [list(extract_hrv_features(extract_rr_intervals(X_raw[idx_test[i % len(idx_test)]])[1]).values())]))

def sens_spec(y_true, y_hat):
    y_true, y_hat = np.asarray(y_true), np.asarray(y_hat)
    return (y_hat[y_true == 1] == 1).mean(), (y_hat[y_true == 0] == 0).mean()

rows = [('HRV features + Random Forest', rf_acc, rf_auc, *sens_spec(y_test, y_pred), f'{rf_nodes:,} tree nodes', rf_train_time, rf_ms)]
if HAS_TF:
    cnn_acc = accuracy_score(y_test, cnn_pred)
    cnn_auc = roc_auc_score(y_test, cnn_proba)
    cnn_ms = median_ms(lambda i: cnn(Xc_test[i % len(Xc_test)][np.newaxis], training=False))
    rows.append(('1D CNN (raw ECG)', cnn_acc, cnn_auc, *sens_spec(y_test, cnn_pred), f'{cnn.count_params():,} parameters', cnn_train_time, cnn_ms))

print(f"{'Model':<30} {'Accuracy':>9} {'AUROC':>7} {'Sens':>6} {'Spec':>6} {'Size':>22} {'Train (s)':>10} {'Infer (ms/ECG)':>15}")
print("-" * 111)
for name, acc, auc_, se, sp, size, tt, ms in rows:
    print(f"{name:<30} {acc:>9.3f} {auc_:>7.3f} {se:>6.1%} {sp:>6.1%} {size:>22} {tt:>10.2f} {ms:>15.2f}")
print(f"{'Chapter 3 wearable (clinical validation)':<40} {'':>6} {'95.0%':>6} {'98.0%':>6}")

# Interpretability: the forest tells us which features drove its decisions
imp = sorted(zip(feature_names, rf.feature_importances_), key=lambda x: -x[1])[:3]
print("\n🔍 Interpretability: top Random Forest features: " + ", ".join(f"{n} ({v:.2f})" for n, v in imp))
print("   The CNN has no built-in equivalent; it would need saliency methods (Chapter 10).")

if HAS_TF:
    diff = cnn_acc - rf_acc
    if abs(diff) < 0.01:
        verdict = "about as accurate as"
    elif diff > 0:
        verdict = f"{diff:.1%} MORE accurate than"
    else:
        verdict = f"{-diff:.1%} LESS accurate than"
    print(f"\n💡 On this synthetic data the 1D CNN is {verdict} the HRV-feature Random Forest,")
    print(f"   took {cnn_train_time / max(rf_train_time, 1e-9):.0f}× longer to train, and is "
          f"{cnn_ms / rf_ms:.1f}× {'slower' if cnn_ms > rf_ms else 'faster'} per ECG at inference.")
    if diff <= 0.01:
        print("   When the signal is rhythm irregularity, hand-crafted HRV features already capture it;")
        print("   the CNN's advantage appears with subtler morphology and much more training data.")
else:
    print("\nℹ️ CNN comparison skipped (no TensorFlow); the feature-based classifier is the browser result.")


## 4. The Prevalence Problem

Even with 98% specificity, screening low-risk populations yields poor PPV.

In [5]:
def calculate_ppv(sensitivity, specificity, prevalence):
    """Calculate positive predictive value."""
    tp = sensitivity * prevalence
    fp = (1 - specificity) * (1 - prevalence)
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_ratio = fp / tp if tp > 0 else float('inf')
    return ppv, fp_ratio

# From confusion matrix
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"Random Forest performance (test set, recorded at rest like clinical validation data):")
print(f"  Sensitivity: {sensitivity:.1%}   (Chapter 3 device: 95%)")
print(f"  Specificity: {specificity:.1%}   (Chapter 3 device: 98%)")
ppv_ours, fpr_ours = calculate_ppv(sensitivity, specificity, 0.005)
print(f"  → at Yuki's 0.5% prevalence: PPV {ppv_ours:.0%}, {fpr_ours:.1f} false alarms per true AFib (Chapter 3: ≈19%, ≈4)")
print(f"  (only {tn + fp} normal test ECGs: each false positive moves specificity by {1/(tn + fp):.1%})")

# Calculate PPV for different settings
settings = [
    ('Hospital (symptomatic)', 0.10),
    ('Yuki: 55 y, low-risk (0.5%)', 0.005),   # Chapter 3 base-rate example
    ('Low-risk, lower bound (0.1%)', 0.001)    # Chapter 3 range: 0.1-0.5%
]

print("\n" + "="*70)
print("POSITIVE PREDICTIVE VALUE BY SETTING")
print("="*70)
for setting, prev in settings:
    ppv, fp_ratio = calculate_ppv(sensitivity, specificity, prev)
    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")
print("="*70)

if specificity == 1.0:
    print(f"\n⚠️  Our model made no false positives on {tn + fp} normal test ECGs, so PPV looks perfect.")
    print("   That reflects easy synthetic data and a small test set, not a real-world 0% false-alarm rate.")

# Illustrative wearable with realistic, imperfect performance
ill_sens, ill_spec = 0.95, 0.98
print(f"\nIllustrative device (sensitivity {ill_sens:.0%}, specificity {ill_spec:.0%}):")
for setting, prev in settings:
    ppv, fp_ratio = calculate_ppv(ill_sens, ill_spec, prev)
    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")

ppv_yuki, fp_ratio_yuki = calculate_ppv(ill_sens, ill_spec, 0.005)
n_pop = 10_000
print(f"\nChapter 3 arithmetic for {n_pop:,} people like Yuki (prevalence 0.5%):")
print(f"   {n_pop*0.005:.0f} have AFib → {n_pop*0.005*ill_sens:.1f} detected; "
      f"{n_pop*0.995:,.0f} healthy → {n_pop*0.995*(1-ill_spec):.0f} false alarms")
print("\n⚠️  CLINICAL INSIGHT:")
print("   Same model, different settings → vastly different clinical utility")
print(f"   For Yuki (prevalence 0.5%), {ill_sens:.0%} sensitivity and {ill_spec:.0%} specificity → PPV {ppv_yuki:.0%}, "
      f"≈{fp_ratio_yuki:.0f} false alarms per true AFib ({1-ppv_yuki:.0%} of alerts are false)")

Model Performance (clinical setting):
  Sensitivity: 100.0%
  Specificity: 100.0%

POSITIVE PREDICTIVE VALUE BY SETTING
Hospital (symptomatic)         | Prevalence:  10.0% | PPV: 100.0% | FP ratio:   0.0:1
Screening (general)            | Prevalence:   0.5% | PPV: 100.0% | FP ratio:   0.0:1
Yuki (young, fit)              | Prevalence:   0.1% | PPV: 100.0% | FP ratio:   0.0:1

⚠️  Our model made no false positives on 75 normal test ECGs, so PPV looks perfect.
   That reflects easy synthetic data and a small test set, not a real-world 0% false-alarm rate.

Illustrative device (sensitivity 95%, specificity 98%):
Hospital (symptomatic)         | Prevalence:  10.0% | PPV:  84.1% | FP ratio:   0.2:1
Screening (general)            | Prevalence:   0.5% | PPV:  19.3% | FP ratio:   4.2:1
Yuki (young, fit)              | Prevalence:   0.1% | PPV:   4.5% | FP ratio:  21.0:1

⚠️  CLINICAL INSIGHT:
   Same model, different settings → vastly different clinical utility
   For Yuki (prevalence 0.1%), e

## 5. Motion Artifacts

Wearable devices are especially susceptible to motion artifacts during exercise. Our classifier was trained and tested on recordings made at rest. What happens when people wear it on a morning jog, like Yuki?

To answer that we need **both** kinds of recording under motion: healthy ones (does **specificity** drop?) and AFib ones (does **sensitivity** change?). The PPV during exercise is then computed from the sensitivity and specificity measured *during exercise*, not borrowed from the resting test set. Recordings the pipeline cannot read are reported separately instead of silently counting as "no alert".

In [ ]:
# Example: the same kind of normal ECG, at rest and during a jog
t_motion, ecg_clean = generate_synthetic_ecg(duration=10, rhythm='normal', ectopic_prob=0)
ecg_motion = add_motion_artifact(ecg_clean, intensity=0.3)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(t_motion, ecg_clean, 'b-', linewidth=1)
axes[0].set_title('Clean ECG (Normal Sinus Rhythm, synthetic)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_motion, ecg_motion, 'orange', linewidth=1)
axes[1].set_title('ECG with Motion Artifact (e.g., during exercise)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_afib, ecg_afib, 'r-', linewidth=1)
axes[2].set_title('True AFib', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Time (seconds)')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Field test: 200 healthy AND 200 AFib 30 s recordings, each measured at rest and during a jog.
# During the jog, arm movement corrupts a random 10-40% of each 30 s recording.
# Signal quality: this simple pipeline has no dedicated quality gate. A recording is "unreadable"
# only if too few R peaks are found to compute HRV features. Unreadable recordings are reported
# separately and EXCLUDED from sensitivity/specificity (they are not counted as "no alert").
n_field = 200
field = {}  # (rhythm, condition) -> (alerts among evaluable recordings, number unreadable)
for rhythm in ['normal', 'afib']:
    feats = {'rest': [], 'jog': []}
    for _ in range(n_field):
        _, e = generate_synthetic_ecg(duration=30, rhythm=rhythm)
        seg = int(np.random.uniform(0.1, 0.4) * len(e)); start = np.random.randint(0, len(e) - seg)
        e_jog = add_motion_artifact(e, intensity=np.random.uniform(0.05, 0.2), segment=(start, start + seg))
        for sig, which in [(e, 'rest'), (e_jog, 'jog')]:
            f = extract_hrv_features(extract_rr_intervals(sig)[1])
            if f:
                feats[which].append(list(f.values()))
    for which, rows_ in feats.items():
        alerts = rf.predict(np.array(rows_)) == 1 if rows_ else np.array([], bool)
        field[rhythm, which] = (alerts, n_field - len(rows_))

def field_summary(which):
    """Sensitivity and specificity among EVALUABLE recordings, plus the share that was unreadable."""
    (a_af, u_af), (a_no, u_no) = field['afib', which], field['normal', which]
    sens = a_af.mean() if len(a_af) else float('nan')
    spec = 1 - a_no.mean() if len(a_no) else float('nan')
    return sens, spec, (u_af + u_no) / (2 * n_field)

(sens_rest, spec_rest, unread_rest), (sens_jog, spec_jog, unread_jog) = field_summary('rest'), field_summary('jog')
ppv_rest, fa_rest = calculate_ppv(sens_rest, spec_rest, 0.005)
ppv_jog, fa_jog = calculate_ppv(sens_jog, spec_jog, 0.005)

print(f"Field test on {n_field} healthy + {n_field} AFib synthetic recordings (same recordings, with and without motion):")
print(f"{'Condition':<10} {'Sensitivity':>12} {'Specificity':>12} {'Unreadable':>11} {'False alerts':>13} {'Missed AFib':>12} {'PPV @0.5%':>10} {'FA per true':>12}")
for which, se, sp, un, ppv_, fa_ in [('At rest', sens_rest, spec_rest, unread_rest, ppv_rest, fa_rest),
                                     ('Jogging', sens_jog, spec_jog, unread_jog, ppv_jog, fa_jog)]:
    key = 'rest' if which == 'At rest' else 'jog'
    a_af, a_no = field['afib', key][0], field['normal', key][0]
    print(f"{which:<10} {se:>12.1%} {sp:>12.1%} {un:>11.1%} {f'{a_no.sum()}/{len(a_no)}':>13} {f'{(~a_af).sum()}/{len(a_af)}':>12} {ppv_:>10.1%} {fa_:>12.1f}")
print("Sensitivity and specificity are computed among evaluable (readable) recordings only.")
print("PPV uses the sensitivity AND specificity measured in the same condition, at Yuki's 0.5% prevalence (Chapter 3).")

if unread_jog == 0:
    print("ℹ️  No recording was unreadable: without a signal-quality gate, every motion-corrupted ECG still gets a rhythm label.")
else:
    print(f"ℹ️  {unread_jog:.1%} of jogging recordings were unreadable (too few R peaks) and are excluded above.")
if spec_jog < spec_rest:
    print(f"⚠️  Specificity falls from {spec_rest:.1%} at rest to {spec_jog:.1%} while jogging: motion artifact distorts")
    print("   R-peak detection, so the R-R intervals of healthy people look irregular, like AFib.")
else:
    print("ℹ️  In this run motion did not increase false alarms in healthy recordings; try a higher intensity.")
if sens_jog < sens_rest - 0.02:
    print(f"⚠️  Sensitivity also falls ({sens_rest:.1%} → {sens_jog:.1%}): motion hides some true AFib episodes.")
elif sens_jog > sens_rest + 0.02:
    print(f"ℹ️  Sensitivity rises ({sens_rest:.1%} → {sens_jog:.1%}), but not because the detector got better:")
    print("   motion makes MORE recordings look irregular, AFib or not, so more of everything is flagged.")
else:
    print(f"ℹ️  Sensitivity is about the same ({sens_rest:.1%} at rest, {sens_jog:.1%} jogging): AFib already looks irregular.")
if ppv_jog < ppv_rest:
    print(f"   Net effect at 0.5% prevalence: PPV {ppv_rest:.0%} at rest vs {ppv_jog:.1%} during exercise "
          f"({fa_rest:.0f} vs {fa_jog:.0f} false alarms per true AFib).")
    print("   Performance measured at rest overstates real-world performance during daily activity.")
else:
    print(f"   Net effect at 0.5% prevalence: PPV {ppv_rest:.1%} at rest vs {ppv_jog:.1%} during exercise: no loss in this run.")

## 6. Key Takeaways

### What We Learned

1. **ECG rhythm classification**: Can use hand-crafted features (HRV) or deep learning (1D CNN). Section 3b printed which was more accurate and how much more compute the CNN needs on this synthetic data; with irregular R-R intervals as the main signal, simple HRV features are hard to beat

2. **Prevalence matters enormously**:
   - For the illustrative device in Section 4 (95% sensitivity, 98% specificity):
     Hospital (10% AFib): PPV ~84%; Yuki's setting (0.5% AFib): PPV ~19%, about 4 false alarms per true alert; 0.1% AFib: PPV ~4.5%
   - Compare our own classifier's sensitivity/specificity and PPV printed in Section 4: the synthetic data was calibrated to land near the published 95%/98% at rest, but with only 150 normal test ECGs a single false positive moves specificity by 0.7%
   - Same model, vastly different utility

3. **Motion artifacts**: Major challenge for wearable devices
   - Exercise, arm movement mimics irregular rhythm
   - Leads to false positives: Section 5 measured sensitivity and specificity at rest vs. during a jog on the same healthy and AFib recordings (PPV during exercise uses both), and counted unreadable recordings separately

4. **Clinical implications**:
   - Wearable AFib detection useful for high-risk individuals
   - Should not be used for mass screening of low-risk populations
   - Need confirmatory testing pathways

5. **Yuki's experience** (Journey 2): Multiple false alarms caused anxiety, unnecessary testing, and ultimately loss of trust in technology

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Yuki's story provides the clinical motivation
- **Chapter 5 (Evaluation)**: Prevalence dependence of PPV, metrics in imbalanced settings
- **Chapter 7 (Time Series)**: This chapter - signal processing, frequency domain features, 1D CNNs
- **Chapter 11 (Deployment)**: Deployment considerations, user experience, regulatory pathways

### Real-World Devices

- Apple Watch, Fitbit, KardiaMobile: FDA-cleared for AFib detection
- Clinical guidelines: Use with confirmatory testing, not as standalone diagnostic
- Appropriate for: Patients with symptoms or known AFib for monitoring

---

## Exercises

1. Implement frequency domain features (FFT, power spectral density)
2. Train the 1D CNN (Section 3b) with motion-artifact augmentation (Section 5) and compare its robustness with the HRV classifier
3. Simulate different prevalence settings and calculate required specificity for PPV >50%
4. Add varying levels of motion artifact and measure degradation

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 7: Time Series Analysis)*  
*This implements Journey 2 (Yuki - AFib Detection) from Chapter 3*  
*For clinical context, see Chapter 3. For time series methods, see Chapter 7.*